## Kanana-1.5-V 3B 유형별 LoRA + 검증 매핑 RAG

44점대 Kanana Shared→MC·SA·LA 유형별 LoRA 조건을 유지하고, 사람이 직접 승인한
대회 이미지↔TourAPI 문서 매핑에만 RAG 설명을 적용하는 비교 실험입니다.

- 최종 매핑 DB: `rag_mapping.jsonl` 2,000건
- RAG 적용: 승인된 train 3건, validation 2건, test 7건만
- 미매칭 문항: 기존 이미지+질문 프롬프트 그대로 처리
- 검색 재실행 없음: BGE·DINO·CLIP·Qdrant 모델을 다시 로드하지 않음
- OCR: 이번 RAG 단독 효과 비교에서는 적용하지 않음

첫 실험은 **RAG만 변경한 A/B 비교**입니다. 결과 확인 후 별도 실험에서 선택적
PaddleOCR crop을 추가해야 RAG와 OCR 각각의 영향을 해석할 수 있습니다.

Colab에서 `런타임 → 런타임 유형 변경 → GPU`를 선택한 뒤 위에서부터 실행하세요.
L4(24GB) 이상을 권장합니다.

### 0. Colab 런타임 준비

In [ ]:
import os
import signal
import subprocess
import sys
from pathlib import Path


restart_marker = Path("/content/.kanana15v_verified_mapping_rag_env_v1")

if not restart_marker.exists():
    def run_pip(*args):
        subprocess.check_call([sys.executable, "-m", "pip", *args])

    run_pip("uninstall", "-q", "-y", "gradio", "gradio_client", "torchao", "torchaudio", "Pillow")
    run_pip(
        "install", "-q", "-U",
        "transformers>=4.57.0,<5",
        "huggingface-hub==0.36.2",
        "accelerate>=1.2",
        "peft>=0.14",
        "datasets>=3.2",
        "timm>=1.0",
        "omegaconf>=2.3",
        "sentencepiece",
        "safetensors",
        "pandas>=2.2",
        "tqdm>=4.66",
    )
    run_pip("install", "-q", "--no-cache-dir", "--force-reinstall", "Pillow==11.3.0")
    restart_marker.write_text("ready\n", encoding="utf-8")
    print("패키지 설치 완료. 런타임을 자동 재시작합니다.")
    print("재연결된 뒤 '모두 실행'을 다시 누르세요.")
    os.kill(os.getpid(), signal.SIGKILL)

import PIL
from PIL import Image, ImageOps

print("Pillow 정상 로드:", PIL.__version__)

### 1. Drive 연결 및 실행 설정

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import copy
import gc
import json
import os
import random
import re
import shutil
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from tqdm.auto import tqdm


# Google Drive 경로
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/CV_korea")
DRIVE_ZIP_DIR = DRIVE_PROJECT_DIR / "data"
RAG_DB_DIR = DRIVE_PROJECT_DIR / "tourapi_image_db"
MAPPING_DB_PATH = (
    RAG_DB_DIR / "mapping" / "combined" / "final" / "rag_mapping.jsonl"
)

MODEL_ID = "kakaocorp/kanana-1.5-v-3b-instruct"
# 현재 Transformers의 Qwen2-VL position_embeddings 호출 방식과 호환되는 revision
KANANA_REVISION = "2e00ef13ccec2e99459a8eada18a1bfd05bff44b"

RUN_NAME = "kanana15v_3b_lora_verified_mapping_rag_typed"
TRAINING_METHOD = "shared_then_typed_lora_with_manual_verified_mapping_rag"

# True: Shared + 유형별 Adapter 학습 / False: 같은 RUN_NAME에 저장된 Adapter로 추론만 수행
DO_TRAIN = True
REUSE_COMPLETED_ADAPTERS = True

TRAIN_MAX_SAMPLES = None
VALIDATION_MAX_SAMPLES = 200
SHARED_EPOCHS = 1
TYPED_EPOCHS = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 16
TRAIN_MAX_IMAGE_SIDE = 896
INFERENCE_MAX_IMAGE_SIDE = 1344
MAX_SEQUENCE_LENGTH = 4096

# 승인된 문서 하나만 사용하며 너무 긴 설명은 잘라냅니다.
RAG_TOP_K = 1
RAG_DOC_CHAR_LIMIT = 900
RESET_PREDICTION_CACHE = False
SAVE_EVERY_N_PREDICTIONS = 20
SEED = 42
REFERENCE_BASELINE_SCORE = 44.8395439
OCR_MODE = "disabled_for_rag_ablation"

COLAB_WORK_DIR = Path("/content/kculture_kanana15v_verified_mapping_rag")
EXTRACT_DIR = COLAB_WORK_DIR / "extracted"
CHECKPOINT_ROOT = DRIVE_PROJECT_DIR / "checkpoints" / RUN_NAME
RUN_OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs" / RUN_NAME
SHARED_ADAPTER_DIR = RUN_OUTPUT_DIR / "adapter_shared"
FORM_ADAPTER_DIRS = {
    form: RUN_OUTPUT_DIR / f"adapter_{form.lower()}" for form in ["MC", "SA", "LA"]
}

for path in [COLAB_WORK_DIR, EXTRACT_DIR, CHECKPOINT_ROOT, RUN_OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("대회 ZIP 폴더:", DRIVE_ZIP_DIR)
print("최종 매핑 DB:", MAPPING_DB_PATH)
print("결과 폴더:", RUN_OUTPUT_DIR)
print("모델:", MODEL_ID)
print("RAG: 사람이 승인한 동일 대상 문서 1개만 선택 적용")
print("OCR:", OCR_MODE)
print("학습: Shared", SHARED_EPOCHS, "epoch + 유형별", TYPED_EPOCHS, "epoch")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 아닙니다. Colab 런타임 유형을 GPU로 변경하세요.")

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU: {gpu_name}")
print(f"VRAM: {gpu_mem_gb:.1f} GiB")
print(f"BF16 지원: {bf16_supported}")

if DO_TRAIN and gpu_mem_gb < 20:
    raise RuntimeError(
        "Kanana-V 3B 일반 LoRA에는 20GB 이상의 VRAM을 권장합니다. "
        "Colab L4 또는 A100 런타임으로 변경하세요."
    )

print("GPU 점검 통과")

### 2. ZIP 자동 탐색 및 압축 해제

In [ ]:
EXPECTED_ARCHIVES = [
    "한국문화 멀티모달 질의응답.zip",
    "train.zip",
    "validation.zip",
    "test.zip",
]


def normalized_name(text):
    return unicodedata.normalize("NFC", str(text))


def resolve_archive(directory, expected_name):
    expected_nfc = normalized_name(expected_name)
    matches = [
        path
        for path in directory.glob("*.zip")
        if normalized_name(path.name) == expected_nfc
    ]
    if len(matches) != 1:
        found = [path.name for path in directory.glob("*.zip")]
        raise FileNotFoundError(
            f"{expected_name!r}을 찾지 못했습니다. 현재 ZIP 목록: {found}"
        )
    return matches[0]


archive_paths = [resolve_archive(DRIVE_ZIP_DIR, name) for name in EXPECTED_ARCHIVES]
archive_state = {
    path.name: {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
    for path in archive_paths
}
state_path = EXTRACT_DIR / ".archive_state.json"

previous_state = None
if state_path.exists():
    previous_state = json.loads(state_path.read_text(encoding="utf-8"))

if previous_state == archive_state:
    print("동일한 ZIP이 이미 해제되어 있어 압축 해제를 건너뜁니다.")
else:
    print("ZIP 4개를 Colab 로컬 디스크에 해제합니다.")
    for archive_path in archive_paths:
        print("-", archive_path.name)
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(EXTRACT_DIR)
    state_path.write_text(
        json.dumps(archive_state, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("압축 해제 완료:", EXTRACT_DIR)

### 3. JSON·이미지 경로 탐색 및 무결성 검사

In [ ]:
def find_split_json(split):
    matches = [
        path
        for path in EXTRACT_DIR.rglob("*.json")
        if normalized_name(path.stem).endswith(f"_{split}")
    ]
    if len(matches) != 1:
        raise RuntimeError(f"{split} JSON 후보가 1개가 아닙니다: {matches}")
    return matches[0]


def read_json(path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


json_paths = {split: find_split_json(split) for split in ["train", "validation", "test"]}
records = {split: read_json(path) for split, path in json_paths.items()}


def find_split_image_dir(split, split_records):
    sample_names = [row["model_input"]["image_name"] for row in split_records[:10]]
    scores = Counter()
    for image_name in sample_names:
        for candidate in EXTRACT_DIR.rglob(image_name):
            if candidate.is_file():
                scores[candidate.parent] += 1
    if not scores:
        raise FileNotFoundError(f"{split} 이미지 디렉터리를 찾지 못했습니다.")
    best_dir, score = scores.most_common(1)[0]
    if score < min(3, len(sample_names)):
        raise RuntimeError(f"{split} 이미지 경로 탐색 결과가 불안정합니다: {scores}")
    return best_dir


image_dirs = {
    split: find_split_image_dir(split, split_records)
    for split, split_records in records.items()
}

expected_counts = {"train": 1000, "validation": 200, "test": 800}
summary_rows = []
for split, split_records in records.items():
    image_dir = image_dirs[split]
    missing = [
        row["model_input"]["image_name"]
        for row in split_records
        if not (image_dir / row["model_input"]["image_name"]).is_file()
    ]
    forms = Counter(row["metadata"]["question_form"] for row in split_records)
    assert len(split_records) == expected_counts[split], (split, len(split_records))
    assert not missing, f"{split} 누락 이미지 {len(missing)}개: {missing[:5]}"
    if split == "test":
        assert all("model_output" not in row for row in split_records)
    else:
        assert all(isinstance(row["model_output"]["answer"], str) for row in split_records)
    summary_rows.append(
        {
            "split": split,
            "rows": len(split_records),
            "MC": forms["MC"],
            "SA": forms["SA"],
            "LA": forms["LA"],
            "image_dir": str(image_dir),
            "missing_images": len(missing),
        }
    )

display(pd.DataFrame(summary_rows))
print("JSON 경로")
for split, path in json_paths.items():
    print(f"- {split}: {path}")

### 4. 최종 이미지↔TourAPI 매핑 DB 검증

In [ ]:
def atomic_write_json(data, path):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def read_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            value = json.loads(line)
            if not isinstance(value, dict):
                raise RuntimeError(f"{path}:{line_number}가 JSON 객체가 아닙니다.")
            rows.append(value)
    return rows


if not MAPPING_DB_PATH.is_file():
    raise FileNotFoundError(
        "최종 매핑 DB가 없습니다. 다음 위치에 rag_mapping.jsonl을 업로드하세요:\n"
        + str(MAPPING_DB_PATH)
    )

mapping_rows = read_jsonl(MAPPING_DB_PATH)
if len(mapping_rows) != 2000:
    raise RuntimeError(f"매핑 DB가 2,000건이 아닙니다: {len(mapping_rows)}")

mapping_by_key = {}
for row in mapping_rows:
    key = str(row.get("image_key", ""))
    if not key or key in mapping_by_key:
        raise RuntimeError(f"매핑 DB image_key가 비어 있거나 중복됩니다: {key}")
    mapping_by_key[key] = row

dataset_keys = {
    f"{split}:{record['model_input']['image_name']}"
    for split in ["train", "validation", "test"]
    for record in records[split]
}
if dataset_keys != set(mapping_by_key):
    missing = sorted(dataset_keys - set(mapping_by_key))
    extra = sorted(set(mapping_by_key) - dataset_keys)
    raise RuntimeError(f"대회 데이터와 매핑 키가 다릅니다. missing={missing[:5]}, extra={extra[:5]}")

matched_rows = [row for row in mapping_rows if row.get("mapping_status") == "matched"]
unmatched_rows = [row for row in mapping_rows if row.get("mapping_status") == "unmatched"]
if len(matched_rows) + len(unmatched_rows) != len(mapping_rows):
    raise RuntimeError("matched/unmatched 이외의 mapping_status가 있습니다.")
if any(not row.get("doc_id") or not row.get("description") for row in matched_rows):
    raise RuntimeError("matched 행의 doc_id 또는 description이 비어 있습니다.")
if any(row.get("doc_id") is not None for row in unmatched_rows):
    raise RuntimeError("unmatched 행에 doc_id가 남아 있습니다.")

mapping_status_rows = []
for split in ["train", "validation", "test"]:
    split_rows = [row for row in mapping_rows if row["split"] == split]
    mapping_status_rows.append({
        "split": split,
        "rows": len(split_rows),
        "rag_matched": sum(row["mapping_status"] == "matched" for row in split_rows),
        "unmatched": sum(row["mapping_status"] == "unmatched" for row in split_rows),
    })

display(pd.DataFrame(mapping_status_rows))
print("최종 매핑 DB 검증 완료")
print("matched:", len(matched_rows), "/", len(mapping_rows))

### 5. Train·Validation·Test 직접 매핑 RAG 구성

In [ ]:
def mapping_key_for(record, split):
    return f"{split}:{record['model_input']['image_name']}"


def rag_item_for(record, split):
    key = mapping_key_for(record, split)
    mapping = mapping_by_key[key]
    if mapping["mapping_status"] != "matched":
        return {
            "rag_applied": False,
            "mapping_status": "unmatched",
            "documents": [],
        }
    document = {
        "doc_id": str(mapping["doc_id"]),
        "title": str(mapping.get("title", "")),
        "description": str(mapping.get("description", "")),
        "image_path": str(mapping.get("rag_image_path", "")),
        "image_url": str(mapping.get("rag_image_url", "")),
        "source": str(mapping.get("source", "tourapi")),
        "confidence": str(mapping.get("confidence", "verified")),
    }
    return {
        "rag_applied": True,
        "mapping_status": "matched",
        "documents": [document],
    }


def rag_context_for(record, split):
    item = rag_item_for(record, split)
    if not item["rag_applied"]:
        return ""
    document = item["documents"][0]
    description = document["description"].strip()[:RAG_DOC_CHAR_LIMIT]
    return "\n".join([
        "[검증된 참고자료]",
        f"제목: {document['title']}",
        f"설명: {description}",
        f"출처: {document['source']}",
    ])


rag_summary_rows = []
for split in ["train", "validation", "test"]:
    split_items = [rag_item_for(record, split) for record in records[split]]
    rag_summary_rows.append({
        "split": split,
        "rows": len(split_items),
        "rag_applied": sum(item["rag_applied"] for item in split_items),
        "rag_not_applied": sum(not item["rag_applied"] for item in split_items),
    })

display(pd.DataFrame(rag_summary_rows))
print("임베딩·검색 없이 최종 매핑 DB를 직접 사용합니다.")

### 6. 검증된 RAG 근거를 포함한 유형별 프롬프트

In [ ]:
SYSTEM_PROMPT = (
    "당신은 이미지와 검증된 참고자료를 함께 이해하는 한국문화 질의응답 모델입니다. "
    "이미지와 질문을 먼저 확인하고, 제공된 참고자료가 답에 필요한 경우 활용하세요. "
    "사용자가 요구한 정답 형식만 출력하고 풀이 과정은 출력하지 마세요."
)


def build_user_text(record, split, retry=False):
    form = record["metadata"]["question_form"]
    model_input = record["model_input"]
    question = model_input["question"].strip()
    options = model_input.get("options") or []
    rag_context = rag_context_for(record, split)

    parts = []
    if rag_context:
        parts.append(
            "다음 자료는 사진과 같은 대상으로 사람이 확인한 참고자료입니다. "
            "질문에 필요한 정보만 사용하세요.\n\n" + rag_context
        )
    parts.append("질문:\n" + question)
    if options:
        parts.append("선택지:\n" + "\n".join(str(option) for option in options))

    if form == "MC":
        instruction = (
            "선다형입니다. 정답 선택지 번호만 출력하세요. "
            "복수 정답이면 번호를 오름차순으로 /로 연결하세요(예: 1/3/5). "
            "정답 이외의 설명은 쓰지 마세요."
        )
    elif form == "SA":
        instruction = (
            "단답형입니다. 질문에 명시된 음절·글자·어절 수를 지키고 정답만 출력하세요. "
            "복수 요소를 요구하면 질문의 순서대로 /로 연결하세요."
        )
    elif form == "LA":
        instruction = (
            "서술형입니다. 이미지 정보와 필요한 한국문화 지식을 반영해 한 문장 이상, "
            "250자 이하로 답하세요. 답변 본문만 출력하세요."
        )
    else:
        raise ValueError(f"알 수 없는 question_form: {form}")

    if retry:
        instruction += " 이전 출력 형식이 잘못되었습니다. 이번에는 반드시 지정 형식만 출력하세요."
    parts.append(instruction)
    return "\n\n".join(parts)


def build_conversation(record, split, image_count, answer=None, add_answer=False, retry=False):
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": " ".join(["<image>"] * image_count)},
        {"role": "user", "content": build_user_text(record, split, retry=retry)},
    ]
    if add_answer:
        conversation.append({"role": "assistant", "content": str(answer)})
    return conversation


for form in ["MC", "SA", "LA"]:
    form_records = [
        row for row in records["train"]
        if row["metadata"]["question_form"] == form
    ]
    example = next(
        (row for row in form_records if rag_item_for(row, "train")["rag_applied"]),
        form_records[0],
    )
    print("=" * 80)
    print(form, example["metadata"]["question_id"])
    print("RAG 적용:", rag_item_for(example, "train")["rag_applied"])
    print(build_user_text(example, "train"))
    print("정답:", example["model_output"]["answer"])

### 7. Kanana-1.5-V 3B BF16/FP16 로드

In [ ]:
from transformers import AutoProcessor
from transformers.dynamic_module_utils import get_class_from_dynamic_module


compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=KANANA_REVISION,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token


def load_base_model():
    # Kanana 공식 remote code는 내부 vision encoder를 FlashAttention2로 먼저
    # 만들도록 고정되어 있습니다. Colab 기본 환경에는 flash_attn이 없으므로,
    # 동적 모델 클래스를 먼저 불러온 뒤 vision encoder의 _from_config 호출만
    # PyTorch SDPA로 고정합니다. 모델 forward 코드는 공식 revision 그대로 씁니다.
    model_class = get_class_from_dynamic_module(
        "modeling.KananaVForConditionalGeneration",
        MODEL_ID,
        revision=KANANA_REVISION,
    )
    vision_class = model_class.__init__.__globals__["CustomQwen2VLVE"]
    original_from_config = vision_class._from_config.__func__

    @classmethod
    def from_config_with_sdpa(cls, config, **kwargs):
        kwargs["attn_implementation"] = "sdpa"
        return original_from_config(cls, config, **kwargs)

    vision_class._from_config = from_config_with_sdpa
    return model_class.from_pretrained(
        MODEL_ID,
        revision=KANANA_REVISION,
        dtype=compute_dtype,
        device_map={"": 0},
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",
    )


def set_model_cache(enabled, target_model=None):
    target = target_model if target_model is not None else model
    if hasattr(target.config, "use_cache"):
        target.config.use_cache = enabled
    if hasattr(target.config, "text_config"):
        target.config.text_config.use_cache = enabled
    language_model = getattr(target, "language_model", None)
    if language_model is not None and hasattr(language_model, "config"):
        language_model.config.use_cache = enabled


gc.collect()
torch.cuda.empty_cache()
model = load_base_model()
set_model_cache(not DO_TRAIN)
torch.set_float32_matmul_precision("high")

print("모델 로드 완료:", MODEL_ID)
print("Kanana revision:", KANANA_REVISION)
print("학습 방식: 일반 LoRA (양자화 없음)")
print("원본 모델 dtype:", compute_dtype)
print("Attention backend: PyTorch SDPA")
print("모델 입력 장치:", next(model.parameters()).device)

### 8. 이미지 로더·응답 정규화·Kanana 추론 함수

In [ ]:
Image.MAX_IMAGE_PIXELS = None


def image_path_for(record, split):
    return image_dirs[split] / record["model_input"]["image_name"]


def load_rgb_image(path, max_side=None):
    with Image.open(path) as opened:
        image = ImageOps.exif_transpose(opened).convert("RGB")
        if max_side is not None and max(image.size) > max_side:
            image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
        return image.copy()


def build_input_images(record, split, max_image_side):
    # 첫 실험에서는 대회 원본 이미지만 Kanana에 넣습니다.
    # TourAPI 이미지 벡터는 관련 설명을 검색하는 데 사용됩니다.
    return [load_rgb_image(image_path_for(record, split), max_image_side)]


def strip_thinking(text):
    text = str(text).strip()
    if "</think>" in text:
        text = text.split("</think>")[-1]
    text = re.sub(r"<think>.*?</think>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    return text.strip()


def normalize_mc(text):
    cleaned = strip_thinking(text)
    cleaned = re.sub(r"^(정답|답|answer)\s*[:：]?\s*", "", cleaned, flags=re.IGNORECASE)
    first_line = next((line.strip() for line in cleaned.splitlines() if line.strip()), "")
    match = re.search(r"(?<!\d)([1-5](?:\s*[/,]\s*[1-5])*)(?!\d)", first_line)
    if not match:
        return "", False
    numbers = sorted(set(re.findall(r"[1-5]", match.group(1))), key=int)
    answer = "/".join(numbers)
    return answer, bool(re.fullmatch(r"[1-5](?:/[1-5])*", answer))


def normalize_short_answer(text):
    cleaned = strip_thinking(text)
    cleaned = re.sub(r"^(정답|답|answer)\s*[:：]?\s*", "", cleaned, flags=re.IGNORECASE)
    first_line = next((line.strip() for line in cleaned.splitlines() if line.strip()), "")
    first_line = re.sub(r"\s+", " ", first_line).strip(" \t\"'“”‘’")
    return first_line.rstrip("。")


def normalize_long_answer(text):
    cleaned = strip_thinking(text)
    cleaned = re.sub(r"^(정답|답변|answer)\s*[:：]?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" \t\"'“”‘’")
    return cleaned[:250].rstrip()


def normalize_prediction(text, form):
    if form == "MC":
        return normalize_mc(text)
    if form == "SA":
        answer = normalize_short_answer(text)
        return answer, bool(answer)
    if form == "LA":
        answer = normalize_long_answer(text)
        return answer, bool(answer) and len(answer) <= 250
    raise ValueError(form)


def max_new_tokens_for(form):
    return {"MC": 16, "SA": 64, "LA": 256}[form]


def move_to_device(value, device):
    if isinstance(value, torch.Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move_to_device(item, device) for key, item in value.items()}
    if isinstance(value, list):
        return [move_to_device(item, device) for item in value]
    return value


INFERENCE_USES_TYPED_ADAPTERS = False
ACTIVE_ADAPTER_NAME = None


def activate_adapter(form):
    global ACTIVE_ADAPTER_NAME
    if not INFERENCE_USES_TYPED_ADAPTERS:
        return
    if ACTIVE_ADAPTER_NAME != form:
        model.language_model.set_adapter(form)
        ACTIVE_ADAPTER_NAME = form


@torch.inference_mode()
def generate_raw(record, split, retry=False):
    form = record["metadata"]["question_form"]
    activate_adapter(form)
    images = build_input_images(record, split, INFERENCE_MAX_IMAGE_SIDE)
    sample = {
        "image": images,
        "conv": build_conversation(record, split, len(images), retry=retry),
    }
    inputs = processor.batch_encode_collate(
        [sample],
        padding_side="left",
        add_generation_prompt=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )
    device = next(model.parameters()).device
    inputs = move_to_device(inputs, device)
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens_for(form),
        temperature=0,
        top_p=1.0,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )
    raw_text = processor.tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()
    del images, sample, inputs, generated
    return raw_text


def predict_one(record, split):
    form = record["metadata"]["question_form"]
    raw_text = generate_raw(record, split, retry=False)
    answer, valid = normalize_prediction(raw_text, form)
    retried = False

    if not valid:
        retried = True
        retry_raw = generate_raw(record, split, retry=True)
        retry_answer, retry_valid = normalize_prediction(retry_raw, form)
        raw_text = raw_text + "\n[RETRY]\n" + retry_raw
        if retry_valid:
            answer, valid = retry_answer, True

    if not valid:
        answer = "1" if form == "MC" else "확인 불가"

    return {"answer": answer, "raw": raw_text, "retried": retried, "valid": valid}

### 모델 입출력 스모크 테스트

In [ ]:
smoke_record = next(
    record for record in records["validation"]
    if rag_item_for(record, "validation")["rag_applied"]
)
smoke_rag = rag_item_for(smoke_record, "validation")

print("ID:", smoke_record["metadata"]["question_id"])
print("질문:", smoke_record["model_input"]["question"])
print("정답:", smoke_record["model_output"]["answer"])
print("RAG 적용:", smoke_rag["rag_applied"])
print("RAG 문서:", [doc["title"] for doc in smoke_rag["documents"]])

smoke_result = predict_one(smoke_record, "validation")
print("예측:", smoke_result["answer"])
print("원본 출력:", smoke_result["raw"])

### 9. Shared LoRA 및 유형별 추가 학습 준비

In [ ]:
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import Dataset


class MultimodalQADataset(Dataset):
    def __init__(self, split_records, max_samples=None, question_form=None):
        rows = list(split_records)
        if question_form is not None:
            rows = [row for row in rows if row["metadata"]["question_form"] == question_form]
        if max_samples is not None:
            rows = rows[:max_samples]
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


class KananaVLCollator:
    def __init__(self, processor, split, max_length, max_image_side):
        self.processor = processor
        self.split = split
        self.max_length = max_length
        self.max_image_side = max_image_side

    def __call__(self, features):
        if len(features) != 1:
            raise RuntimeError("Kanana-V collator는 per_device_train_batch_size=1을 전제로 합니다.")

        record = features[0]
        images = build_input_images(record, self.split, self.max_image_side)
        answer = record["model_output"]["answer"]

        full_sample = {
            "image": images,
            "conv": build_conversation(
                record, self.split, len(images), answer=answer, add_answer=True
            ),
        }
        prompt_sample = {
            "image": images,
            "conv": build_conversation(record, self.split, len(images), add_answer=False),
        }

        full_batch = self.processor.batch_encode_collate(
            [full_sample],
            padding_side="right",
            add_generation_prompt=False,
            max_length=self.max_length,
        )
        prompt_batch = self.processor.batch_encode_collate(
            [prompt_sample],
            padding_side="right",
            add_generation_prompt=True,
            max_length=self.max_length,
        )

        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        labels[full_batch["input_ids"] < 0] = -100
        prompt_length = int(prompt_batch["attention_mask"].sum().item())
        labels[:, :prompt_length] = -100
        if torch.all(labels == -100):
            raise RuntimeError("정답 토큰이 학습 label에 포함되지 않았습니다.")

        full_batch["labels"] = labels
        return full_batch


train_dataset = MultimodalQADataset(records["train"], TRAIN_MAX_SAMPLES)
typed_train_datasets = {
    form: MultimodalQADataset(records["train"], question_form=form)
    for form in ["MC", "SA", "LA"]
}
train_collator = KananaVLCollator(
    processor=processor,
    split="train",
    max_length=MAX_SEQUENCE_LENGTH,
    max_image_side=TRAIN_MAX_IMAGE_SIDE,
)


def available_language_lora_targets(language_model):
    preferred = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    targets = [
        name
        for name, module in language_model.named_modules()
        if isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in preferred
        and "lm_head" not in name
    ]
    if not targets:
        raise RuntimeError("Kanana language_model의 LoRA target module을 찾지 못했습니다.")
    return targets


target_modules = available_language_lora_targets(model.language_model)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

if DO_TRAIN:
    for parameter in model.parameters():
        parameter.requires_grad = False
    # Kanana-V 전체가 아니라 내부 Llama language_model만 PEFT로 감쌉니다.
    # 원본 Kanana 멀티모달 forward/generate와 vision encoder는 그대로 유지됩니다.
    model.language_model = get_peft_model(model.language_model, lora_config)
    model.language_model.print_trainable_parameters()
    print("학습 방식: Shared LoRA → MC·SA·LA 유형별 LoRA")
    print("LoRA target 수:", len(target_modules))
    print("유형별 학습 문항 수:", {form: len(ds) for form, ds in typed_train_datasets.items()})
else:
    print("DO_TRAIN=False: 저장된 MC·SA·LA Adapter로 추론합니다.")

### 학습 전 loss 스모크 테스트

In [ ]:
if DO_TRAIN:
    model.train()
    one_batch = train_collator([train_dataset[0]])
    device = next(model.parameters()).device
    one_batch = move_to_device(one_batch, device)
    smoke_output = model(**one_batch)
    print("학습 loss 스모크 테스트:", float(smoke_output.loss))
    smoke_output.loss.backward()
    grad_count = sum(
        parameter.grad is not None
        for parameter in model.parameters()
        if parameter.requires_grad
    )
    if grad_count == 0:
        raise RuntimeError("LoRA parameter에 gradient가 생성되지 않았습니다.")
    model.zero_grad(set_to_none=True)
    print("gradient가 확인된 trainable tensor 수:", grad_count)
    del one_batch, smoke_output
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("추론 모드이므로 학습 loss 테스트를 건너뜁니다.")

### 10. Shared LoRA 학습 후 MC·SA·LA Adapter 분화

In [ ]:
from transformers import Trainer, TrainingArguments


def adapter_ready(path):
    return (path / "adapter_config.json").exists()


def run_training_stage(stage_model, dataset, stage_name, epochs):
    checkpoint_dir = CHECKPOINT_ROOT / stage_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    set_model_cache(False, stage_model)

    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        optim="adamw_torch",
        logging_steps=10,
        # 전체 Kanana-V를 checkpoint로 저장하면 수 GB가 되므로 단계 완료 후 LoRA만 저장합니다.
        save_strategy="no",
        bf16=bf16_supported,
        fp16=not bf16_supported,
        gradient_checkpointing=False,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(
        model=stage_model,
        args=training_args,
        train_dataset=dataset,
        data_collator=train_collator,
    )
    result = trainer.train()
    return trainer, result.metrics


stage_metrics = {}

if DO_TRAIN:
    # 1단계: 전체 데이터 Shared Adapter
    if REUSE_COMPLETED_ADAPTERS and adapter_ready(SHARED_ADAPTER_DIR):
        print("완료된 Shared Adapter를 재사용합니다:", SHARED_ADAPTER_DIR)
        stage_metrics["shared"] = {"reused": True}
    else:
        trainer, metrics = run_training_stage(
            model, train_dataset, "shared", SHARED_EPOCHS
        )
        model.language_model.save_pretrained(str(SHARED_ADAPTER_DIR))
        processor.save_pretrained(str(SHARED_ADAPTER_DIR))
        stage_metrics["shared"] = metrics
        print("Shared Adapter 저장:", SHARED_ADAPTER_DIR)
        del trainer

    del model
    gc.collect()
    torch.cuda.empty_cache()

    # 2단계: Shared Adapter 가중치에서 시작해 유형별로 독립 추가 학습
    for form in ["MC", "SA", "LA"]:
        form_dir = FORM_ADAPTER_DIRS[form]
        if REUSE_COMPLETED_ADAPTERS and adapter_ready(form_dir):
            print(f"완료된 {form} Adapter를 재사용합니다:", form_dir)
            stage_metrics[form] = {"reused": True}
            continue

        typed_model = load_base_model()
        typed_model.language_model = PeftModel.from_pretrained(
            typed_model.language_model,
            str(SHARED_ADAPTER_DIR),
            is_trainable=True,
        )
        trainer, metrics = run_training_stage(
            typed_model,
            typed_train_datasets[form],
            f"typed_{form.lower()}",
            TYPED_EPOCHS,
        )
        typed_model.language_model.save_pretrained(str(form_dir))
        stage_metrics[form] = metrics
        print(f"{form} Adapter 저장:", form_dir)

        del trainer, typed_model
        gc.collect()
        torch.cuda.empty_cache()

    train_metrics_path = RUN_OUTPUT_DIR / "train_metrics.json"
    train_metrics_path.write_text(
        json.dumps(stage_metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("학습 지표 저장:", train_metrics_path)
else:
    missing = [str(path) for path in FORM_ADAPTER_DIRS.values() if not adapter_ready(path)]
    if missing:
        raise FileNotFoundError("저장된 유형별 Adapter가 없습니다: " + " | ".join(missing))
    del model
    gc.collect()
    torch.cuda.empty_cache()


# validation/test에서 문항 유형에 따라 Adapter만 빠르게 전환합니다.
model = load_base_model()
model.language_model = PeftModel.from_pretrained(
    model.language_model,
    str(FORM_ADAPTER_DIRS["MC"]),
    adapter_name="MC",
    is_trainable=False,
)
for form in ["SA", "LA"]:
    model.language_model.load_adapter(
        str(FORM_ADAPTER_DIRS[form]),
        adapter_name=form,
        is_trainable=False,
    )

model.eval()
set_model_cache(True, model)
INFERENCE_USES_TYPED_ADAPTERS = True
ACTIVE_ADAPTER_NAME = None
gc.collect()
torch.cuda.empty_cache()

print("유형별 Adapter 추론 준비 완료:", list(FORM_ADAPTER_DIRS))

### 11. Validation 직접 매핑 RAG 진단

유형과 RAG 적용 여부별 exact match를 확인합니다. 매칭 validation은 2건뿐이므로
개별 예측도 함께 확인하고, 최종 판단은 리더보드 점수로 진행합니다.

In [ ]:
def comparable_answer(text, form):
    if form == "MC":
        normalized, valid = normalize_mc(text)
        return normalized if valid else str(text).strip()
    return re.sub(r"\s+", " ", str(text)).strip()


validation_subset = records["validation"][:VALIDATION_MAX_SAMPLES]
validation_rows = []

for record in tqdm(validation_subset, desc="validation 검증 매핑 RAG 추론"):
    result = predict_one(record, "validation")
    form = record["metadata"]["question_form"]
    target = record["model_output"]["answer"]
    rag_item = rag_item_for(record, "validation")
    validation_rows.append({
        "question_id": record["metadata"]["question_id"],
        "question_form": form,
        "rag_applied": rag_item["rag_applied"],
        "rag_doc_ids": "/".join(doc["doc_id"] for doc in rag_item["documents"]),
        "rag_titles": " / ".join(doc["title"] for doc in rag_item["documents"]),
        "prediction": result["answer"],
        "target": target,
        "exact_match": comparable_answer(result["answer"], form)
        == comparable_answer(target, form),
        "prediction_length": len(result["answer"]),
        "retried": result["retried"],
        "raw": result["raw"],
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df.groupby(["question_form", "rag_applied"])["exact_match"].agg(["count", "mean"]))
display(validation_df[validation_df["rag_applied"]])

validation_path = RUN_OUTPUT_DIR / "validation_predictions.csv"
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")
print("validation 결과 저장:", validation_path)

### 12. Test 800건 검증 매핑 RAG 추론

test 7건에만 승인된 TourAPI 설명이 들어가며 나머지 793건은 기존 프롬프트를 사용합니다.
모델, Adapter, 매핑 DB 또는 프롬프트를 변경하면 `RUN_NAME`을 바꾸거나
`RESET_PREDICTION_CACHE=True`로 한 번 실행하세요.

In [ ]:
prediction_cache_path = RUN_OUTPUT_DIR / "test_prediction_cache.json"


if RESET_PREDICTION_CACHE and prediction_cache_path.exists():
    prediction_cache_path.unlink()
    print("기존 test 예측 캐시를 삭제했습니다.")

if prediction_cache_path.exists():
    prediction_cache = json.loads(prediction_cache_path.read_text(encoding="utf-8"))
else:
    prediction_cache = {}

test_ids = {row["metadata"]["question_id"] for row in records["test"]}
prediction_cache = {
    question_id: value
    for question_id, value in prediction_cache.items()
    if question_id in test_ids
}

print(f"재사용 가능한 기존 예측: {len(prediction_cache)} / {len(records['test'])}")

completed_since_save = 0
for record in tqdm(records["test"], desc="test 검증 매핑 RAG 추론"):
    question_id = record["metadata"]["question_id"]
    if question_id in prediction_cache:
        continue

    result = predict_one(record, "test")
    rag_item = rag_item_for(record, "test")
    prediction_cache[question_id] = {
        "answer": result["answer"],
        "raw": result["raw"],
        "question_form": record["metadata"]["question_form"],
        "rag_applied": rag_item["rag_applied"],
        "rag_doc_ids": [doc["doc_id"] for doc in rag_item["documents"]],
        "rag_titles": [doc["title"] for doc in rag_item["documents"]],
        "retried": result["retried"],
        "valid": result["valid"],
    }
    completed_since_save += 1

    if completed_since_save >= SAVE_EVERY_N_PREDICTIONS:
        atomic_write_json(prediction_cache, prediction_cache_path)
        completed_since_save = 0

atomic_write_json(prediction_cache, prediction_cache_path)
print("test 예측 완료:", len(prediction_cache))
print("예측 캐시:", prediction_cache_path)

### 13. 제출 JSON·ZIP 생성 및 최종 검증

In [ ]:
if len(prediction_cache) != len(records["test"]):
    missing_ids = [
        row["metadata"]["question_id"]
        for row in records["test"]
        if row["metadata"]["question_id"] not in prediction_cache
    ]
    raise RuntimeError(f"예측이 완료되지 않은 test 문항: {missing_ids[:10]}")

submission = copy.deepcopy(records["test"])
preview_rows = []

for record in submission:
    question_id = record["metadata"]["question_id"]
    form = record["metadata"]["question_form"]
    cached = prediction_cache[question_id]
    answer = str(cached["answer"]).strip()
    record["model_output"] = {"answer": answer}
    preview_rows.append(
        {
            "question_id": question_id,
            "question_form": form,
            "image_name": record["model_input"]["image_name"],
            "rag_applied": cached["rag_applied"],
            "rag_titles": " / ".join(cached["rag_titles"]),
            "answer": answer,
            "answer_length": len(answer),
        }
    )

assert len(submission) == 800
assert [row["metadata"]["question_id"] for row in submission] == [
    row["metadata"]["question_id"] for row in records["test"]
]
for original, completed in zip(records["test"], submission):
    assert original["metadata"] == completed["metadata"]
    assert original["model_input"] == completed["model_input"]
    answer = completed["model_output"]["answer"]
    form = completed["metadata"]["question_form"]
    assert isinstance(answer, str) and answer.strip()
    if form == "MC":
        assert re.fullmatch(r"[1-5](?:/[1-5])*", answer), (form, answer)
    if form == "LA":
        assert len(answer) <= 250, (form, len(answer))

submission_json_path = RUN_OUTPUT_DIR / "submission_kanana15v_3b_lora_verified_mapping_rag_typed.json"
submission_zip_path = RUN_OUTPUT_DIR / "submission_kanana15v_3b_lora_verified_mapping_rag_typed.zip"
submission_preview_path = RUN_OUTPUT_DIR / "submission_preview.csv"

atomic_write_json(submission, submission_json_path)
pd.DataFrame(preview_rows).to_csv(
    submission_preview_path, index=False, encoding="utf-8-sig"
)
with zipfile.ZipFile(submission_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(submission_json_path, arcname=submission_json_path.name)

reloaded_submission = json.loads(submission_json_path.read_text(encoding="utf-8"))
assert len(reloaded_submission) == 800
assert all("model_output" in row for row in reloaded_submission)

print("제출 파일 생성 완료")
print("JSON:", submission_json_path)
print("ZIP :", submission_zip_path)
print("미리보기:", submission_preview_path)
display(pd.DataFrame(preview_rows).groupby(["question_form", "rag_applied"]).size())
display(pd.DataFrame(preview_rows).head(10))

### 14. 검증 매핑 RAG 실험 요약 저장

In [ ]:
rag_experiment_summary = {
    "run_name": RUN_NAME,
    "model_id": MODEL_ID,
    "reference_baseline_score": REFERENCE_BASELINE_SCORE,
    "mapping_db": str(MAPPING_DB_PATH),
    "mapping_record_count": len(mapping_rows),
    "mapping_matched_count": len(matched_rows),
    "retrieval_summary": rag_summary_rows,
    "train_method": TRAINING_METHOD,
    "shared_epochs": SHARED_EPOCHS,
    "typed_epochs": TYPED_EPOCHS,
    "ocr_mode": OCR_MODE,
    "submission_json": str(submission_json_path),
    "submission_zip": str(submission_zip_path),
    "validation_csv": str(validation_path),
}

rag_experiment_summary_path = RUN_OUTPUT_DIR / "rag_experiment_summary.json"
atomic_write_json(rag_experiment_summary, rag_experiment_summary_path)
print(json.dumps(rag_experiment_summary, ensure_ascii=False, indent=2))
print("RAG 실험 요약:", rag_experiment_summary_path)

### 실행 결과 확인 순서

1. 매핑 DB가 전체 2,000건, matched 12건인지 확인
2. 적용 수가 train=3, validation=2, test=7인지 확인
3. 학습 전 loss와 LoRA gradient 스모크 테스트 통과 확인
4. Shared 1 epoch + MC·SA·LA 유형별 1 epoch 완료 확인
5. validation에서 RAG 적용 2건의 개별 예측 확인
6. `submission_kanana15v_3b_lora_verified_mapping_rag_typed.json` 제출
7. 44.8395 기준 점수와 비교

이번 노트북에는 OCR이 적용되지 않습니다. RAG 단독 결과가 확인된 다음 동일 조건에서
선택적 PaddleOCR crop만 추가해야 두 변화의 효과를 분리해서 비교할 수 있습니다.